# ARC-AGI-3 DQN: paired Colab CPU / T4 benchmark

Open this notebook in https://colab.research.google.com via **File → Upload notebook**. Select **Runtime → Change runtime type → T4 GPU**, save, then **Runtime → Run all**. Upload the accompanying `dqn_colab_source_*.zip` when prompted. GPU availability is not guaranteed. The notebook refuses to label another GPU as T4.

This is a short **V0 shared-wrapper training** benchmark, not frozen inference or legacy native training. Source comes from the current unpushed `integration/2026-10-24` working tree, not a nonexistent remote branch. No GitHub login, clone, push, or existing checkpoint is needed. The bundle contains code and the already exposed `ls20` offline runtime cache, with per-file SHA256 provenance; no historical checkpoints, results, credentials or heldout games. Do not inspect game source to design the policy.

Two fresh runs: **seed42, 2,000 selected actions each, batch8, warmup64, train-every5, decay2000, V0, full-pixel action map, level resets, episode-limit0, one CPU thread**. Each should perform **388 updates** (first at65, last at2000). CPU and CUDA RNG trajectories can diverge numerically; same protocol does not guarantee identical actions. No long training is scheduled.


In [ ]:
import sys, subprocess, platform, os, json, hashlib, zipfile, uuid, shutil, time
from pathlib import Path
assert sys.version_info >= (3,12), "Select a Colab Python 3.12+ runtime for these pinned dependencies"
# Keep Colab's CUDA-enabled PyTorch; do not install the CPU PyTorch wheel.
subprocess.run([sys.executable, "-m", "pip", "install", "arc-agi==0.9.9", "arcengine==0.9.3",
                "gymnasium==1.4.0", "numpy==2.5.4", "PyYAML==6.0.3", "pytest>=8,<10"], check=True)
import torch
assert torch.cuda.is_available(), "No CUDA runtime: choose T4 GPU and reconnect"
assert tuple(map(int, torch.__version__.split("+")[0].split(".")[:2])) >= (2,8), "PyTorch >=2.8 required"
assert int(torch.__version__.split('.')[0]) < 3
GPU = torch.cuda.get_device_name(0)
assert "T4" in GPU, f"Expected T4, received {GPU}; change runtime or explicitly document different hardware"
subprocess.run(["nvidia-smi"], check=True)
print(platform.platform(), sys.version, torch.__version__, torch.version.cuda, GPU)


In [ ]:
from google.colab import files
uploaded = files.upload()  # Select exactly the source ZIP delivered with this notebook.
assert len(uploaded) == 1
name, raw = next(iter(uploaded.items()))
assert name.endswith(".zip")
ROOT = Path("/content") / ("arc_dqn_bench_" + uuid.uuid4().hex[:12])
ROOT.mkdir(exist_ok=False)
with zipfile.ZipFile(__import__('io').BytesIO(raw)) as z:
    for member in z.namelist():
        target = (ROOT / member).resolve()
        assert target.is_relative_to(ROOT.resolve()), "Unsafe archive path"
    z.extractall(ROOT)
manifest = json.loads((ROOT / "BENCHMARK_SOURCE_MANIFEST.json").read_text())
for path, expected in manifest["sha256"].items():
    assert hashlib.sha256((ROOT/path).read_bytes()).hexdigest() == expected, path
print("Bundle SHA256:", hashlib.sha256(raw).hexdigest(), "Base commit:", manifest["base_commit"])
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_dqn_shared_training.py", "tests/test_dqn_devices.py"], check=True)


## Persist each completed run
Mount Drive before measuring; training writes to the VM disk and copies to Drive **after** each run. Drive traffic is excluded from throughput. If Drive cannot mount, set `USE_DRIVE=False` and download the ZIP in the last cell before disconnecting. Checkpoints are frozen snapshots with model/target/optimizer/config, **not exact-resume checkpoints**: replay/RNG are not saved. CPU loading of CUDA snapshots requires `map_location="cpu"`.


In [ ]:
USE_DRIVE = True
RUN_ID = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()) + "_" + uuid.uuid4().hex[:8]
OUT = ROOT / "benchmark_outputs" / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
DRIVE_OUT = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUT = Path("/content/drive/MyDrive/ARC_DQN_benchmarks") / RUN_ID
    DRIVE_OUT.mkdir(parents=True, exist_ok=False)
shutil.copy2(ROOT / "BENCHMARK_SOURCE_MANIFEST.json", OUT)
(OUT / "hardware.json").write_text(json.dumps({"platform": platform.platform(), "python": sys.version,
    "gpu": GPU, "cpu": platform.processor(), "cpu_count": os.cpu_count(),
    "torch": torch.__version__, "cuda": torch.version.cuda}, indent=2))
with (OUT / "pip_freeze.txt").open("w") as f:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=f, check=True)


In [ ]:
# Identical commands apart from device and unique suffix, each in a fresh process.
# Stage synchronization gives attributable wall times but adds overhead.
STEPS = 2000  # 2000 or 3000 only; never 450K here.
assert STEPS in (2000, 3000)
PROFILE = True
ORDER = ["cpu", "cuda"]
records = []
for device in ORDER:
    suffix = f"colab_{RUN_ID}_{device}"
    cmd = [sys.executable, "-m", "DQN.train", "--backend", "shared", "--operation-mode", "offline",
           "--game", "ls20-9607627b", "--seed", "42", "--steps", str(STEPS), "--variant", "v0",
           "--batch-size", "8", "--warmup", "64", "--train-every", "5", "--epsilon-decay", "2000",
           "--episode-limit", "0", "--reset-mode", "level", "--torch-threads", "1",
           "--device", device, "--run-suffix", suffix, "--output-root", str(OUT)]
    if PROFILE:
        cmd.append("--profile")
    env = {**os.environ, "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1", "PYTHONHASHSEED": "42"}
    start = time.perf_counter()
    with (OUT / f"{device}_console.txt").open("x") as log:
        completed = subprocess.run(cmd, env=env, stdout=log, stderr=subprocess.STDOUT)
    if completed.returncode:
        print((OUT / f"{device}_console.txt").read_text()[-6000:])
        if DRIVE_OUT:
            shutil.copytree(OUT, DRIVE_OUT, dirs_exist_ok=True)
        raise RuntimeError(f"{device} benchmark failed; see console log")
    process_seconds = time.perf_counter() - start
    matches = list((OUT / "Logs/DQN").glob(f"*_{suffix}_timing.json"))
    assert len(matches) == 1
    record = json.loads(matches[0].read_text())
    expected_updates = STEPS//5 - 12
    assert record["selected_actions"] == STEPS and record["gradient_updates"] == expected_updates
    assert record["official_actions"] == STEPS
    record.update(device=device, process_seconds_including_import_setup_save=process_seconds)
    records.append(record)
    (OUT / "paired_summary.json").write_text(json.dumps(records, indent=2))
    if DRIVE_OUT:
        shutil.copytree(OUT, DRIVE_OUT, dirs_exist_ok=True)
    print(device, "actions/s:", round(record["actions_per_second"], 2),
          "updates:", record["gradient_updates"], "loop seconds:", round(record["loop_seconds"], 2))
print("All outputs:", OUT, "Drive:", DRIVE_OUT)


In [ ]:
import csv
rows = []
for r in records:
    stages = r["stages_seconds"]
    # Nested batch timing must not be added again to the outer gradient timing.
    accounted = sum(stages.get(k, 0) for k in ["environment_step", "environment_reset", "action_selection", "replay_insert", "gradient_update"])
    rows.append({"device": r["device"], "actions/s": r["actions_per_second"], "loop_s": r["loop_seconds"],
                 "env_step_s": stages.get("environment_step", 0), "reset_s": stages.get("environment_reset", 0),
                 "selection_s": stages.get("action_selection", 0),
                 "policy_forward_s_nested": stages.get("policy_forward", 0),
                 "labels_masks_transfer_s_nested": stages.get("gradient_labels_masks_transfer", 0), "gradient_s": stages.get("gradient_update", 0),
                 "batch_encode_transfer_s_nested": stages.get("gradient_batch_encode_transfer", 0),
                 "other_loop_s": r["loop_seconds"] - accounted,
                 "versus_local_24.38_actions_s": r["actions_per_second"] / (10000/410.1400718)})
for row in rows:
    print(json.dumps(row, indent=2))
with (OUT / "comparison.csv").open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)
cpu, gpu = (next(r for r in records if r["device"] == d) for d in ["cpu", "cuda"])
speedup = gpu["actions_per_second"] / cpu["actions_per_second"]
print(f"T4 / Colab CPU throughput ratio: {speedup:.2f}x")
print("Provisional GO" if speedup >= 1.5 else "NO-GO for extended T4 training on speed alone")
print("Confirm a borderline result by repeating in reverse order with PROFILE=False and a NEW RUN_ID.")
print("Environment step+reset and file logging remain CPU work. Batch8 and scalar returns can limit CUDA gains.")
if DRIVE_OUT:
    shutil.copytree(OUT, DRIVE_OUT, dirs_exist_ok=True)
archive = shutil.make_archive(str(OUT), "zip", OUT)
if DRIVE_OUT:
    shutil.copy2(archive, DRIVE_OUT.parent / Path(archive).name)
files.download(archive)


## Interpret before approving longer training

- `loop_seconds` starts after environment/model/history setup and includes selected actions, resets, replay, updates, CSV flushes, progress logging and synchronized timings. `total_seconds_including_checkpoint` adds final save; subprocess time additionally includes imports/setup. Package install, tests, upload, and Drive copy are excluded. No synthetic model-only rate is used as total throughput.
- `action_selection` includes random actions (no network), one-hot encoding, mask upload, forward pass and scalar `.item()` download. Gradient timing includes CPU replay sampling, input/mask/label transfers, target forward, backward, optimizer, target sync and loss scalar download. Nested batch encoding timing covers NumPy stacking, compact uint8 upload, device int64 conversion and PyTorch one-hot; do **not** add it again. `policy_forward` isolates greedy network inference within selection (random actions skip it); `gradient_labels_masks_transfer` isolates action/reward/terminal/mask construction and upload within updates. These nested times are not additive with their parents. Replay stays in host uint8 memory. No AMP, pinned replay, asynchronous copies or vectorized environments change the algorithm here.
- The same shared training path uses `arc_agi` as its underlying local game engine; this is distinct from the legacy `--backend native` trainer and from frozen-policy evaluation. +1 level-delta training reward, controller-owned level RESET, no artificial truncation. RESET is excluded from selected actions; resets are reported separately. Scores/learning quality are not benchmark conclusions.
- The local pilot measured 10,000 / 410.140s = **24.382 actions/s** with 1,988 updates. Short runs have different exploration averages, reset trajectories and startup proportions; use the paired Colab CPU result as the primary hardware comparison. The existing encoding has also been consolidated to transfer uint8 frames before one-hot encoding on the chosen device. Re-run this exact notebook protocol locally for a stronger cross-machine comparison.
- Predeclared provisional rule: T4 throughput **>=1.5× Colab CPU**, finite losses, correct update counts and zero failures merits a repeat with reversed order and `PROFILE=False` in a new output directory. Approve extended training only if the gain persists and is useful against the local24.382 actions/s baseline. **<=1.2×** is a no-go on speed alone; **1.2–1.5×** needs repeat measurement. High environment share suggests another GPU alone will help little. A different NVIDIA PC cannot be ranked without its model and an identical benchmark.
- After confirming speed, agree on the full experimental budget separately. V1/history/V2 throughput and learning gains cannot be inferred from V0. These snapshots do not replace any report models. Run another NVIDIA PC with the same delivered source, dependencies, settings and fresh outputs before moving the workspace.

Colab hardware and session availability vary: https://research.google.com/colaboratory/faq.html . No T4 measurement has been made by merely preparing this notebook.
